# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [ ]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Notas:**
Gymnasium cumple el mismo papel que la clase CheeseWorld en el notebook anterior: es el ambiente/entorno. Provee env.reset(), env.step(action) y env.render(), es decir, define los estados, las acciones válidas, las transiciones y las recompensas, sin decidir nunca qué acción tomar.

El agente, en cambio, no está representado como una clase (como QLearningAgent en CheeseWorld), sino que queda repartido entre una variable y varias funciones:
- La Q-table (Q, un array de numpy) hace las veces del "conocimiento" del agente — es el equivalente a self.Q en CheeseWorld.
- Las funciones sueltas (epsilon_greedy_policy, train_q_learning, greedy_policy) cumplen el rol de los métodos choose_action y update_q, pero en vez de operar sobre self, reciben la Q-table y los hiperparámetros (epsilon, alpha, gamma) como parámetros explícitos en cada llamada.


## 2. Crear FrozenLake


In [ ]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
2. ¿Cuántas columnas?
3. ¿Qué representa una celda \(Q[s,a]\)?

**Respuesta / discusión:**

1. Filas: 16 (una por cada estado del grid 4x4).
2. Columnas: 4 (una por cada acción: Left, Down, Right, Up).
3. ¿Qué representa Q[s,a]? Es la estimación del agente de la recompensa total esperada (acumulada, con descuento γ) si está en el estado s, ejecuta la acción a, y de ahí en adelante sigue actuando de forma óptima. Por ejemplo, Q[5,2] es "qué tan bueno es, estando en el estado 5, moverse hacia la derecha" no es la recompensa inmediata, sino la recompensa esperada de ahí hasta el final del episodio.

In [ ]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

**Notas:**

El 0 no significa que las acciones sean malas, sino que el agente todavía no tiene conocimiento sobre ellas. Como todavía no se ha recorrido la Q-table (no ha habido experiencia real interactuando con el entorno), no hay forma de saber cuál acción es mejor que otra, así que todas parten del mismo punto: cero información.

Cuando varias acciones tienen exactamente el mismo valor máximo (como pasa al inicio, que las 4 valen 0), greedy_policy termina eligiendo una acción al azar entre las que están empatadas. Esto es debido a que si en vez de eso, tomara la primera acción con mayor valor, el agente siempre elegiría la misma acción (Left) en todos los estados al principio, y eso sería un sesgo que no tiene nada que ver con haber aprendido algo, solo con el orden en que están las acciones.


## 3. Ejecutar una transición


In [ ]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$
(0,1,0,4)
$$

El agente partió del estado inicial (0), la política ε-greedy (con ε=1.0, es decir exploración pura) escogió la acción "Down" (acción 1), y el ambiente lo llevó al estado 4 (piso normal, sin peligro), con recompensa 0 porque todavía no llegó a la meta.

¿De cuál de esos cuatro elementos **no disponíamos directamente** en Value Iteration cuando hablábamos de experiencia real?

De los cuatro elementos, los que no teníamos directamente en Value Iteration eran la recompensa (r) y el estado siguiente (s′). En Value Iteration ya conocíamos de antemano la función de transición T(s,a,s′) y la recompensa R(s) sabíamos, sin ejecutar nada, con qué probabilidad se pasaba de un estado a otro y qué recompensa daba cada uno. En Q-Learning, en cambio, el agente no tiene esa tabla de probabilidades: solo puede ejecutar la acción de verdad en el ambiente, y es hasta que "vive" esa transición que descubre cuál fue el estado siguiente y la recompensa recibida. Por eso Q-Learning es model-free: no necesita conocer T ni R de antemano, aprende directamente Q(s,a) acumulando experiencia real, episodio a episodio.

## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [ ]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [ ]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [ ]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.

Lo único que cambia entre los tres momentos es la Q-table (el ambiente, los estados y las acciones posibles son siempre los mismos). Cada vez que el agente ejecuta una acción real y observa (r,s′), se actualiza la ecuacuón.
Este ajuste compara lo que el agente creía que valía esa acción contra lo que realmente pasó (el TD error), y corrige la estimación en esa dirección. Repitiendo esto miles de veces, la Q-table deja de ser ceros y empieza a reflejar qué tan conveniente es cada acción desde cada estado, como una estimación de recompensa acumulada esperada, no una probabilidad.

Relación entre Q_initial, Q_early, Q_trained y las acciones observadas:

- Q_initial (0 episodios): todos los valores están en cero, no hay información aún. El agente actúa prácticamente al azar y en mi simulación terminó cayendo en un hoyo (recompensa total = 0).
- Q_early (50 episodios): todavía casi no hay valores aprendidos en los estados iniciales (siguen en cero), por lo que el comportamiento sigue siendo casi tan errático como el random, en mi simulación también terminó en un hoyo.
- Q_trained (10,000 episodios): la Q-table ya tiene valores bien diferenciados (por ejemplo, Q[0] = [0.735, 0.774, 0.774, 0.735]), lo que le permite al agente elegir siempre la acción con mayor valor esperado. En mi simulación llegó directo a la meta por un camino limpio, con recompensa total = 1.

### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** 0

**Valores Q:**

- Left: 0.735
- Down: 0.774
- Right: 0.774
- Up: 0.735

¿Cuál acción seleccionaría:

$$
\arg\max_a Q(s,a) = acción 1 (Down)
$$

?

**Interpretación:**
El estado 0 es la esquina superior izquierda del grid (S, inicio). Los valores muestran que Down y Right están empatados en 0.774, ambos claramente por encima de Left y Up (0.735). Esto tiene sentido con el mapa: moverse Left o Up desde el estado 0 choca contra el borde y no avanza hacia ningún lado nuevo, mientras que Down y Right sí acercan al agente hacia la meta (estado 15), evitando además el hoyo cercano en el estado 5. argmax_a Q(s,a) devolvió Down porque, al haber empate exacto entre Down y Right, np.argmax() simplemente toma el primer índice con el valor máximo, no lo desempata al azar como sí lo hace la función greedy_policy usada durante el juego real. En la práctica, el agente entrenado elegiría Down o Right indistintamente, ya que ambos representan el mismo nivel de "calidad" aprendida para escapar de la esquina inicial.


## 5. Evaluar la política aprendida


In [ ]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**
Durante el entrenamiento, la exploración (ε-greedy) es necesaria porque el agente todavía no sabe qué acciones son buenas. Sin explorar, nunca descubriría caminos mejores que los primeros que encontró por casualidad. Pero durante la evaluación el objetivo es distinto: ya no se trata de seguir aprendiendo, sino de medir qué tan buena es la política que el agente ya aprendió. Si se siguiera explorando en esta etapa, el agente tomaría acciones subóptimas a propósito una parte del tiempo, lo cual bajaría artificialmente el desempeño medido y no reflejaría el verdadero conocimiento acumulado en la Q-table. Esto se confirma con mi resultado: usando argmax_a Q(s,a) (pura explotación) en los 100 episodios de evaluación, obtuve mean_reward=1.000 y std_reward=0.000, es decir, la política aprendida es completamente consistente y siempre exitosa cuando no se introduce aleatoriedad de exploración.


## 6. Experimento: FrozenLake estocástico


In [ ]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?
Con is_slippery=False, cada acción lleva siempre al mismo estado siguiente (transición determinística). Con is_slippery=True, la acción elegida solo se ejecuta con cierta probabilidad; el resto de las veces el agente se desliza hacia una dirección perpendicular no deseada. El ambiente deja de ser predecible.

¿Qué cambia en la **ecuación de Q-Learning**?
La ecuación no cambia en absoluto: sigue siendo exactamente igual,con los mismos parámetros α y γ. Lo que cambia es que ahora un mismo par (s,a) puede producir distintos (r,s′) cada vez que se ejecuta, porque el resultado depende del azar del ambiente. Por eso hace falta muchos más episodios (20,000 en vez de 10,000) para que la Q-table logre un promedio confiable sobre los posibles resultados futuros, que ya no se pueden predecir con certeza por la aleatoriedad del ambiente.

**Discusión:**

- Ambiente:En mi entrenamiento con is_slippery=False obtuve 100% de éxito consistente (mean=1.0, std=0.0), mientras que con is_slippery=True obtuve solo 6% de éxito (mean=0.06, std=0.237). Al repetir el entrenamiento estocástico varias veces se confirmó que hay mucha variabilidad entre ejecuciones (entre 12% y 70% de éxito en pruebas adicionales), lo cual muestra que la estocasticidad no solo dificulta aprender una buena política, sino que también hace que el resultado del entrenamiento sea menos reproducible.

- Algoritmo: La fórmula de actualización es idéntica en ambos casos; lo único que cambia es la calidad y consistencia de la experiencia (s,a,r,s′) que recibe el agente. Con datos más ruidosos, se necesita mayor cantidad de episodios para que Q(s,a) converja hacia un promedio confiable, en lugar de sobreajustarse a resultados puntuales.

# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**
Q(s,a) almacena una estimación numérica de la recompensa acumulada esperada si el agente está en el estado s, ejecuta la acción a, y de ahí en adelante sigue actuando de forma óptima. 
No es una probabilidad ni la acción a tomar, es solo un número que sirve como punto de comparación: cuando se comparan las Q(s,a) de todas las acciones posibles en un mismo estado (por ejemplo con argmax), ahí sí se puede determinar cuál acción conviene más.

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**
Sale de la fila de la Q-table correspondiente al estado siguiente s′(el estado al que llegó el agente después de ejecutar la acción). De esa fila se toma el valor más alto entre todas las acciones posibles a′ desde ese estado.
Por ejemplo, con la experiencia (s,a,r,s′)=(0,2,0,1), el estado siguiente es s′=1, y su fila en Q_trained es [0.735, 0, 0.815, 0.774]; el máximo es $0.815$ (acción Right). Este valor no viene de env.step() como sí lo hacen 𝑟 y s′: es información que la Q-table ya tenía guardada de episodios anteriores. Se usa el máximo (y no un promedio) porque al actualizar Q(s,a) se asume que, una vez que el agente llegue al estado s′, de ahí en adelante actuará de forma óptima (siguiendo la mejor acción posible), no de cualquier manera.

**3. ¿Por qué necesitamos $\epsilon$-greedy?**
ε-greedy define la probabilidad con la que el agente elige explorar (probar una acción al azar) en vez de explotar (elegir la acción que ya cree que es mejor, según lo que sabe hasta el momento). Se necesita explorar porque, si el agente solo explotara desde el inicio, se quedaría atrapado usando las primeras acciones que le dieron algún resultado, sin descubrir si existían caminos mejores que nunca llegó a probar. Explorando, el agente visita distintos pares (s,a) y así construye una Q-table más completa y confiable, que eventualmente le permite encontrar rutas más óptimas hacia la recompensa.

**4. ¿Por qué Q-Learning es model-free?**
Q-Learning es model-free porque el agente aprende a tomar decisiones mediante prueba y error, ejecutando acciones reales en el ambiente y observando qué pasa, en vez de necesitar de antemano las reglas del entorno (es decir, la función de transición T(s,a,s′) y la función de recompensa R(s)). A diferencia de Value Iteration o Policy Iteration, que sí requieren conocer T y R para calcular V(s) sin ejecutar ninguna acción, Q-Learning solo tiene la Q-table y va actualizándola episodio a episodio con la experiencia (s,a,r,s′) que efectivamente vive, sin nunca modelar explícitamente cómo funciona el ambiente por dentro.